# Title/Abstract Screening with AgenticWorkflow

This notebook demonstrates how to use `TitleAbstractReviewer` with `AgenticWorkflow` to screen a set of articles based on title and abstract content.

We'll cover:
1. Single-round screening with one reviewer
2. Two-round workflow with an expert second-pass reviewer
3. Interpreting the output DataFrame columns

## Setup and Sample Data

In [ ]:
import pandas as pd
from lattereview.agentic import TitleAbstractReviewer, AgenticWorkflow
from pydantic_ai.models.test import TestModel

# Sample dataset of 5 articles for screening
data = pd.DataFrame({
    "title": [
        "Deep Learning for Chest X-Ray Diagnosis: A Multicenter Study",
        "Patient Satisfaction Surveys in Rural Clinics",
        "Transformer Models for ECG Arrhythmia Detection",
        "Hospital Cafeteria Menu Optimization Using Linear Programming",
        "Federated Learning for Multi-Site Medical Image Segmentation",
    ],
    "abstract": [
        "We developed a deep learning model to classify 14 thoracic pathologies from chest X-rays across 5 hospitals. The model achieved AUC > 0.90 for 12 of 14 conditions on external validation.",
        "This study surveyed 200 patients in 10 rural clinics about their satisfaction with telehealth services introduced during the pandemic. Results showed 78% satisfaction rate.",
        "A transformer-based architecture was applied to 12-lead ECG signals for automated arrhythmia detection. Our model detected 8 arrhythmia types with 95% accuracy on the PTB-XL dataset.",
        "We applied integer linear programming to optimize weekly menu planning for a 500-bed hospital cafeteria, reducing food waste by 23% while maintaining nutritional standards.",
        "We propose a federated learning framework for training segmentation models across 12 hospitals without sharing patient data. The approach achieved performance within 2% of centralized training.",
    ],
})

# Combine title and abstract into a single text column for review
data["text"] = data["title"] + "\n\n" + data["abstract"]
print(f"Dataset: {len(data)} articles")
data[["title"]].head()

## Single-Round Screening

We create a `TitleAbstractReviewer` with inclusion/exclusion criteria and run it through `AgenticWorkflow`.

> **Note:** Uses `TestModel` for structure demonstration. For real screening, use `model="openai:gpt-4o"` (requires `OPENAI_API_KEY`).

In [ ]:
screener = TitleAbstractReviewer(
    name="Screener",
    backstory="You are a systematic review expert screening articles on AI in medical imaging.",
    model=TestModel(),
    inclusion_criteria="Studies that apply machine learning or deep learning to medical imaging tasks (X-ray, CT, MRI, ECG, etc.)",
    exclusion_criteria="Studies not involving AI/ML methods; studies focused on hospital operations, surveys, or non-clinical applications",
    max_iterations=1,
)

# Define a single-round workflow
workflow = AgenticWorkflow(
    workflow_schema=[
        {
            "round": "A",
            "reviewers": [screener],
            "text_inputs": ["text"],
        }
    ],
    verbose=True,
)

result_df = await workflow(data)

In [ ]:
# Inspect output columns added by the workflow
output_cols = [c for c in result_df.columns if c.startswith("round-")]
print("Output columns:", output_cols)
print()

# Expected columns: round-A_Screener_decision, round-A_Screener_reasoning, etc.
result_df[["title"] + output_cols].head()

## Two-Round Workflow with Expert Second Pass

In a two-round workflow, the second round can filter on results from the first. For example, a senior expert reviews only items where the first screener was uncertain or included them.

> **Note:** Uses `TestModel`. Replace with real model string for actual screening.

In [ ]:
# First-pass screener
first_screener = TitleAbstractReviewer(
    name="Screener",
    backstory="You are a research assistant performing initial title/abstract screening.",
    model=TestModel(),
    inclusion_criteria="Studies applying AI/ML to medical imaging",
    exclusion_criteria="Non-AI studies, operational studies, surveys",
    max_iterations=1,
)

# Second-pass expert
expert_screener = TitleAbstractReviewer(
    name="Expert",
    backstory="You are a senior radiologist and AI researcher. You carefully evaluate borderline cases.",
    model=TestModel(),
    inclusion_criteria="Studies with rigorous ML methodology applied to diagnostic medical imaging",
    exclusion_criteria="Weak methodology, non-diagnostic applications, pure signal processing without ML",
    max_iterations=1,
)

# Two-round workflow: expert reviews only items included by the first screener
workflow_2 = AgenticWorkflow(
    workflow_schema=[
        {
            "round": "A",
            "reviewers": [first_screener],
            "text_inputs": ["text"],
        },
        {
            "round": "B",
            "reviewers": [expert_screener],
            "text_inputs": ["text", "round-A_Screener_output"],
            "filter": lambda row: str(row.get("round-A_Screener_decision", "")).strip() != "",
        },
    ],
    verbose=True,
)

result_df_2 = await workflow_2(data)

In [ ]:
# View results from both rounds
all_output_cols = [c for c in result_df_2.columns if c.startswith("round-")]
print("All output columns:", all_output_cols)
print()
result_df_2[["title"] + all_output_cols]

## Output Column Naming Convention

The workflow adds columns following this pattern:

| Column | Description |
|--------|-------------|
| `round-A_Screener_decision` | Include/exclude decision from the Screener in round A |
| `round-A_Screener_reasoning` | The reviewer's reasoning for the decision |
| `round-B_Expert_decision` | Decision from the Expert in round B |
| `round-B_Expert_reasoning` | Expert's reasoning |

Items not reviewed in a round (e.g., filtered out) will have `NaN` in that round's columns.